# Dubey & Müller (2019), Figure 1 — full-fidelity reproduction

Reproduces **both panels of Figure 1** as dense power curves at the paper's 500 replications, instead of the three-point check the pytest version runs.

| panel | sweep | what it tests |
|---|---|---|
| left | location `delta` over `[-1, 1]`, `sd = 0.5` | the `F_n` term of eq. (7) |
| right | scale ratio `r` over `[0.125, 3]` at equal means, `sd = 0.2` | the Levene-type `U_n` term of eq. (8) |

The right panel is the one that matters most for us: it is exactly what a between/within-variance proxy **cannot** reproduce, and it is why `frechet_anova` was rewritten to the paper's eqs. (6)–(11).

**Design** (paper Section 5): the random objects are `N(mu, 1)` distributions under the L2-Wasserstein metric. Since `W_2(N(a,1), N(b,1)) = |a - b|`, the space collapses to the real line and the Fréchet mean is the arithmetic mean — so the statistic is evaluated exactly, with no barycentre machinery. `n1 = n2 = 100`, `alpha = 0.05`, mu truncated to `[-10, 10]`.

**Cost**: a few minutes on 32 cores; this design is pure scalar numpy, so memory is negligible.

In [ ]:
# Must run BEFORE numpy is imported anywhere.
import os

for _v in ("OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS",
           "NUMEXPR_NUM_THREADS", "VECLIB_MAXIMUM_THREADS"):
    os.environ[_v] = "1"

print("logical CPUs:", os.cpu_count())
!free -g | head -2

In [ ]:
REPO_URL = "https://github.com/hugogobato/Pointcloud_Equality_Testing.git"
REPO_DIR = "Pointcloud_Equality_Testing"

import os

if not os.path.isdir(REPO_DIR):
    !git clone -q $REPO_URL
else:
    !cd $REPO_DIR && git pull -q

# This notebook never touches persistent homology, but tda2s.repro imports the
# competitor registry, so the PH stack still has to be importable.
!pip install -q gudhi==3.11.0 ripser==0.6.10 persim==0.3.8
!pip install -q --no-deps -e $REPO_DIR

import sys

if os.path.abspath(REPO_DIR) not in sys.path:
    sys.path.insert(0, os.path.abspath(REPO_DIR))
print("ready")

In [ ]:
import json
import time

import numpy as np

from tda2s.repro.parallel import default_workers, run_dubey_muller_sweep

REPS = 500                        # the paper's budget
N_PERM = 200
N_JOBS = default_workers(cap=32)

DELTAS = np.round(np.linspace(-1.0, 1.0, 21), 4)      # Fig. 1 left
RATIOS = np.round(np.linspace(0.125, 3.0, 24), 4)     # Fig. 1 right

print(f"{REPS} reps, {N_PERM} permutations, {N_JOBS} workers")
print(f"left panel: {len(DELTAS)} deltas   right panel: {len(RATIOS)} ratios")

In [ ]:
# ---- Figure 1 left: location shift, sd = 0.5 ---------------------------
t0 = time.perf_counter()
left = run_dubey_muller_sweep(DELTAS, "delta", REPS, base_seed=0,
                              n_jobs=N_JOBS, chunk=50,
                              sd=0.5, n=100, n_perm=N_PERM)
print(f"\nleft panel: {time.perf_counter() - t0:.1f}s")

In [ ]:
# ---- Figure 1 right: scale ratio at equal means, sd = 0.2 --------------
t0 = time.perf_counter()
right = run_dubey_muller_sweep(RATIOS, "r", REPS, base_seed=50000,
                               n_jobs=N_JOBS, chunk=50,
                               delta=0.0, sd=0.2, n=100, n_perm=N_PERM)
print(f"\nright panel: {time.perf_counter() - t0:.1f}s")

In [ ]:
def _row(res, key):
    return {"value": key, "rate": res[key]["rate"], "se": res[key]["se"],
            "n": res[key]["n"]}

left_rows = [_row(left, float(d)) for d in DELTAS]
right_rows = [_row(right, float(r)) for r in RATIOS]

size_left = next(r for r in left_rows if abs(r["value"]) < 1e-9)
size_right = next(r for r in right_rows if abs(r["value"] - 1.0) < 1e-9) \
    if any(abs(r["value"] - 1.0) < 1e-9 for r in right_rows) else None

print(f"level at delta = 0 : {size_left['rate']:.4f} +- {size_left['se']:.4f} "
      f"(nominal 0.05)")
if size_right is not None:
    print(f"level at r = 1     : {size_right['rate']:.4f} +- {size_right['se']:.4f}")
else:
    print("level at r = 1     : r = 1 is not on the grid; see the curve")

print("\nleft panel (delta, rejection rate):")
for r in left_rows:
    print(f"  {r['value']:+.3f}  {r['rate']:.3f} +- {r['se']:.3f}")

os.makedirs("results", exist_ok=True)
OUT = f"results/dubey_muller_figure1_reps{REPS}.json"
with open(OUT, "w") as fh:
    json.dump({"reps": REPS, "n_perm": N_PERM,
               "left_sd": 0.5, "right_sd": 0.2,
               "left": left_rows, "right": right_rows}, fh, indent=2)
print("\nwrote", OUT)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].errorbar([r["value"] for r in left_rows], [r["rate"] for r in left_rows],
               yerr=[r["se"] for r in left_rows], marker="o", ms=3)
ax[0].set_xlabel("location shift delta")
ax[0].set_title("Fig. 1 left: location (sd = 0.5)")

ax[1].errorbar([r["value"] for r in right_rows], [r["rate"] for r in right_rows],
               yerr=[r["se"] for r in right_rows], marker="o", ms=3)
ax[1].axvline(1.0, color="grey", lw=0.8, ls="--")
ax[1].set_xlabel("scale ratio r")
ax[1].set_title("Fig. 1 right: scale (sd = 0.2)")

for a in ax:
    a.axhline(0.05, color="grey", lw=0.8, ls=":")
    a.set_ylabel("rejection rate")
    a.set_ylim(-0.03, 1.03)
fig.tight_layout()
FIG = f"results/dubey_muller_figure1_reps{REPS}.png"
fig.savefig(FIG, dpi=150)
print("wrote", FIG)

In [ ]:
for output_file in (OUT, FIG):
    try:
        from google.colab import files
        files.download(output_file)
        print("Downloaded:", output_file)
    except Exception as e:
        print("(Not on Colab / download skipped):", e)